# VL-JEPA: Vision-Language Joint Embedding Predictive Architecture (2-GPU + Wandb)

## Overview
This notebook implements VL-JEPA, aligning video embeddings from V-JEPA2 with text embeddings using contrastive learning.

**Architecture:**
- **Context Encoder**: V-JEPA2 ViT-L (frozen, 304M params, GPU 0)
- **Predictor**: Llama-3.2-1B layers 8-15 (trainable, 490M params, GPU 1)
- **Target Encoder**: Gemma-2-2B (trainable, 2B params, GPU 1)
- **Loss**: InfoNCE contrastive loss in 2048-dim shared space

**Key Fixes Applied:**
1. Single combined device transfer: `visual.to(device1, dtype=torch.float16)`
2. Proper weight initialization with `xavier_uniform_` before `.half()`
3. Gradient clipping to prevent parameter corruption
4. Full Llama model (not extracted layers) with layers 0-7 frozen
5. Comprehensive Wandb logging for monitoring

In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoVideoProcessor,
    VJEPA2Model,
    LlamaForCausalLM
)
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import numpy as np
import wandb
import time

# Check GPU availability
assert torch.cuda.device_count() >= 2, 'Need at least 2 GPUs'
device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')
print(f'GPU 0: {torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'GPU 1: {torch.cuda.get_device_name(1)} - {torch.cuda.get_device_properties(1).total_memory/1e9:.1f}GB')

In [ ]:
# Cell 2: Wandb Init
wandb.init(
    project='vljepa-training',
    name='vljepa-2gpu-fixed',
    config={
        'architecture': 'VL-JEPA',
        'vision_encoder': 'V-JEPA2 ViT-L (frozen)',
        'predictor': 'Llama-3.2-1B layers 8-15',
        'text_encoder': 'Gemma-2-2B',
        'embed_dim': 2048,
        'batch_size': 4,
        'learning_rate': 1e-4,
        'max_steps': 500,
        'gradient_clip': 1.0,
        'num_frames': 16,
        'gpu_split': 'GPU0: V-JEPA2 | GPU1: Llama+Gemma+projections'
    }
)
print('✓ Wandb initialized')

In [ ]:
# Cell 3: Load V-JEPA2 Vision Encoder (GPU 0, frozen)
print('Loading V-JEPA2 ViT-L on GPU 0...')

model_id = 'facebook/vjepa2-vitl-fpc64-256'
vision_processor = AutoVideoProcessor.from_pretrained(model_id)
vision_model = VJEPA2Model.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device0)

# Freeze all parameters
vision_model.eval()
for param in vision_model.parameters():
    param.requires_grad = False

vision_dim = vision_model.config.hidden_size  # 1024
vision_params = sum(p.numel() for p in vision_model.parameters())

print(f'✓ V-JEPA2 loaded: {vision_dim}-dim, {vision_params/1e6:.1f}M params (frozen, GPU 0)')
wandb.config.update({'vision_dim': vision_dim, 'vision_params': vision_params})

In [ ]:
# Cell 4: Load Llama-3.2-1B Predictor (GPU 1)
print('\nLoading Llama-3.2-1B on GPU 1...')

llama = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    torch_dtype=torch.float16
).to(device1)

# Use full model but freeze first 8 layers (0-7)
predictor = llama.model  # Full LlamaModel
predictor_dim = llama.config.hidden_size  # 2048

# Freeze embedding and first 8 layers
for param in predictor.embed_tokens.parameters():
    param.requires_grad = False
for i in range(8):
    for param in predictor.layers[i].parameters():
        param.requires_grad = False

predictor_params = sum(p.numel() for p in predictor.parameters())
predictor_trainable = sum(p.numel() for p in predictor.parameters() if p.requires_grad)

print(f'✓ Llama loaded: {predictor_dim}-dim')
print(f'  Total: {predictor_params/1e6:.1f}M | Trainable: {predictor_trainable/1e6:.1f}M (layers 8-15)')
wandb.config.update({
    'predictor_dim': predictor_dim,
    'predictor_params': predictor_params,
    'predictor_trainable': predictor_trainable
})

In [ ]:
# Cell 5: Load Gemma-2-2B Text Encoder (GPU 1)
print('\nLoading Gemma-2-2B on GPU 1...')

text_model = AutoModel.from_pretrained(
    'google/gemma-2-2b',
    torch_dtype=torch.float16
).to(device1)
text_dim = text_model.config.hidden_size  # 2304

text_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b')
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

text_params = sum(p.numel() for p in text_model.parameters())

print(f'✓ Gemma loaded: {text_dim}-dim, {text_params/1e6:.1f}M params')
wandb.config.update({'text_dim': text_dim, 'text_params': text_params})

In [ ]:
# Cell 6: Dataset (COCO with memory-mapped streaming)
print('\nLoading dataset...')

try:
    dataset = load_dataset('AlexZigma/msr-vtt', split='train', keep_in_memory=False)
    print(f'✓ MSR-VTT: {len(dataset)} videos')
except:
    print('MSR-VTT unavailable, using COCO...')
    dataset = load_dataset('HuggingFaceM4/COCO', split='train', keep_in_memory=False)
    print(f'✓ COCO: {len(dataset)} images')

class VideoTextDataset(Dataset):
    def __init__(self, hf_dataset, text_tokenizer, vision_processor, num_frames=16):
        self.dataset = hf_dataset
        self.tokenizer = text_tokenizer
        self.processor = vision_processor
        self.num_frames = num_frames
        self.has_image = 'image' in hf_dataset[0]
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Load image as repeated frames
        if self.has_image:
            img = sample['image'].convert('RGB')
            frames = [img] * self.num_frames
        else:
            frames = [Image.new('RGB', (224, 224))] * self.num_frames
        
        # Process for vision model
        pixels = self.processor(videos=frames, return_tensors='pt')['pixel_values_videos']
        
        # Remove batch dimension: [1, T, C, H, W] -> [T, C, H, W]
        while pixels.dim() > 4:
            if pixels.shape[0] == 1:
                pixels = pixels.squeeze(0)
            else:
                break
        
        # Get caption
        if 'caption' in sample:
            caption = sample['caption']
        elif 'sentences' in sample:
            caption = sample['sentences']['raw'][0]
        else:
            caption = 'An image.'
        
        # Tokenize
        tokens = self.tokenizer(
            caption,
            return_tensors='pt',
            padding='max_length',
            max_length=77,
            truncation=True
        )
        
        return {
            'pixels': pixels,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }

train_dataset = VideoTextDataset(dataset, text_tokenizer, vision_processor)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=False,
    drop_last=True
)

print(f'✓ Dataset ready: {len(train_dataset)} samples, {len(train_loader)} batches')
wandb.config.update({'dataset_size': len(train_dataset), 'num_batches': len(train_loader)})

In [ ]:
# Cell 7: VL-JEPA Model (2-GPU with all fixes)
class VLJEPA(nn.Module):
    def __init__(self, vision, predictor, text, embed_dim=2048):
        super().__init__()
        self.vision = vision  # GPU 0
        self.predictor = predictor  # GPU 1 - full LlamaModel
        self.text = text  # GPU 1
        
        # FIX: Proper initialization with xavier_uniform BEFORE .half()
        self.proj_in = nn.Linear(vision_dim, predictor_dim)
        nn.init.xavier_uniform_(self.proj_in.weight)
        nn.init.zeros_(self.proj_in.bias)
        self.proj_in = self.proj_in.to(device1).half()
        
        self.pred_proj = nn.Linear(predictor_dim, embed_dim)
        nn.init.xavier_uniform_(self.pred_proj.weight)
        nn.init.zeros_(self.pred_proj.bias)
        self.pred_proj = self.pred_proj.to(device1).half()
        
        self.text_proj = nn.Linear(text_dim, embed_dim)
        nn.init.xavier_uniform_(self.text_proj.weight)
        nn.init.zeros_(self.text_proj.bias)
        self.text_proj = self.text_proj.to(device1).half()
        
        self.temperature = nn.Parameter(torch.tensor(0.07, dtype=torch.float16).to(device1))
    
    def forward(self, pixels, input_ids, attention_mask):
        # Handle device transfers internally
        pixels = pixels.to(device0)
        input_ids = input_ids.to(device1)
        attention_mask = attention_mask.to(device1)
        
        # Vision encoder (frozen, GPU 0)
        with torch.no_grad():
            v_out = self.vision(pixel_values_videos=pixels)
            if hasattr(v_out, 'pooler_output') and v_out.pooler_output is not None:
                visual = v_out.pooler_output
                visual = visual.unsqueeze(1).expand(-1, pixels.shape[1], -1)
            else:
                visual = v_out.last_hidden_state
                if visual.dim() == 2:
                    visual = visual.unsqueeze(1).expand(-1, pixels.shape[1], -1)
            visual = visual.float()
        
        # CRITICAL FIX: Single combined operation to avoid NaN corruption
        # DO NOT use .to(device1).half() - use combined operation
        visual = visual.to(device1, dtype=torch.float16)
        
        # Project to predictor dim
        h = self.proj_in(visual)
        
        # Predictor (full Llama model)
        predictor_out = self.predictor(
            inputs_embeds=h,
            use_cache=False,
            return_dict=True
        )
        h = predictor_out.last_hidden_state
        
        # Project predictor output
        pred = self.pred_proj(h)
        pred = F.normalize(pred, dim=-1)
        
        # Text encoder (GPU 1)
        text_out = self.text(input_ids=input_ids, attention_mask=attention_mask)
        target = text_out.last_hidden_state
        
        # Project text output
        target = self.text_proj(target)
        target = F.normalize(target, dim=-1)
        
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        # FIX: Ensure mask is on same device
        mask = mask.to(pred.device)
        
        # Pool sequences
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        
        # Normalize
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        # InfoNCE loss with temperature
        temp = self.temperature.clamp(min=0.01, max=1.0)
        sim = torch.matmul(pred_pool, target_pool.T) / temp
        
        labels = torch.arange(sim.shape[0], device=sim.device)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        
        # Return both total loss and components for logging
        return (loss_v2t + loss_t2v) / 2, loss_v2t, loss_t2v, sim

# Create model
model = VLJEPA(vision_model, predictor, text_model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n✓ VL-JEPA model created')
print(f'  Total params: {total_params/1e6:.1f}M')
print(f'  Trainable: {trainable_params/1e6:.1f}M')
print(f'  Frozen: {(total_params-trainable_params)/1e6:.1f}M')
print(f'  GPU 0: V-JEPA2 (~304M frozen)')
print(f'  GPU 1: Llama (~490M) + Gemma (~2048M) + projections')

wandb.config.update({
    'total_params': total_params,
    'trainable_params': trainable_params
})

In [ ]:
# Cell 8: Optimizer
base_lr = 1e-4

# Collect trainable parameters
predictor_params = []
for i in range(8, 16):
    predictor_params.extend(list(model.predictor.layers[i].parameters()))
predictor_params.extend(list(model.pred_proj.parameters()))
predictor_params.extend(list(model.proj_in.parameters()))

text_params = list(model.text.parameters()) + list(model.text_proj.parameters())

# Filter for requires_grad
predictor_params = [p for p in predictor_params if p.requires_grad]
text_params = [p for p in text_params if p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': predictor_params, 'lr': base_lr},
    {'params': text_params, 'lr': base_lr * 0.05},
    {'params': [model.temperature], 'lr': base_lr}
], weight_decay=0.01)

print(f'✓ Optimizer created')
print(f'  Predictor LR: {base_lr}')
print(f'  Text LR: {base_lr * 0.05}')
print(f'  Predictor params: {len(predictor_params)}')
print(f'  Text params: {len(text_params)}')

In [ ]:
# Cell 9: Training Loop with Comprehensive Wandb Logging
max_steps = 500
log_interval = 10

print(f'\nTraining for {max_steps} steps...\n')

model.train()
model.vision.eval()  # Keep V-JEPA2 frozen

global_step = 0
step_times = []

for batch in tqdm(train_loader, total=max_steps):
    if global_step >= max_steps:
        break
    
    step_start = time.time()
    
    pixels = batch['pixels']
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    
    # Forward
    pred, target = model(pixels, input_ids, attention_mask)
    loss, loss_v2t, loss_t2v, sim = model.compute_loss(pred, target, attention_mask)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    
    # Compute gradient norms BEFORE clipping
    grad_norms = {}
    total_norm = 0.0
    for name, param in model.named_parameters():
        if param.grad is not None and param.requires_grad:
            param_norm = param.grad.data.norm(2).item()
            total_norm += param_norm ** 2
            if 'proj_in' in name:
                grad_norms['proj_in'] = param_norm
            elif 'pred_proj' in name:
                grad_norms['pred_proj'] = param_norm
            elif 'text_proj' in name:
                grad_norms['text_proj'] = param_norm
    total_norm = total_norm ** 0.5
    
    # Gradient clipping (CRITICAL FIX)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    # Optimizer step
    optimizer.step()
    
    step_time = time.time() - step_start
    step_times.append(step_time)
    
    global_step += 1
    
    # Wandb logging
    log_dict = {
        'loss/total': loss.item(),
        'loss/v2t': loss_v2t.item(),
        'loss/t2v': loss_t2v.item(),
        'temperature': model.temperature.item(),
        'gradients/total_norm': total_norm,
        'gradients/proj_in_norm': grad_norms.get('proj_in', 0),
        'gradients/pred_proj_norm': grad_norms.get('pred_proj', 0),
        'gradients/text_proj_norm': grad_norms.get('text_proj', 0),
        'timing/step_time': step_time,
        'timing/samples_per_sec': 4 / step_time,  # batch_size=4
        'memory/gpu0_allocated_gb': torch.cuda.memory_allocated(0) / 1e9,
        'memory/gpu1_allocated_gb': torch.cuda.memory_allocated(1) / 1e9,
        'memory/gpu0_reserved_gb': torch.cuda.memory_reserved(0) / 1e9,
        'memory/gpu1_reserved_gb': torch.cuda.memory_reserved(1) / 1e9,
    }
    
    # Similarity statistics (for monitoring alignment quality)
    with torch.no_grad():
        diag_sim = torch.diag(sim)  # Positive pairs
        off_diag_sim = sim[~torch.eye(sim.shape[0], dtype=torch.bool, device=sim.device)]  # Negative pairs
        log_dict.update({
            'similarity/positive_mean': diag_sim.mean().item(),
            'similarity/positive_std': diag_sim.std().item(),
            'similarity/negative_mean': off_diag_sim.mean().item(),
            'similarity/negative_std': off_diag_sim.std().item(),
            'similarity/margin': (diag_sim.mean() - off_diag_sim.mean()).item(),
        })
    
    # Check for NaN (should never happen with fixes)
    has_nan = (
        torch.isnan(model.proj_in.weight).any() or
        torch.isnan(model.pred_proj.weight).any() or
        torch.isnan(model.text_proj.weight).any()
    )
    log_dict['health/has_nan'] = float(has_nan)
    
    wandb.log(log_dict, step=global_step)
    
    # Console logging
    if global_step % log_interval == 0:
        avg_time = np.mean(step_times[-log_interval:])
        print(f'Step {global_step:3d} | Loss: {loss.item():.4f} (v2t: {loss_v2t.item():.4f}, t2v: {loss_t2v.item():.4f}) | '
              f'Temp: {model.temperature.item():.4f} | Grad: {total_norm:.2e} | {avg_time:.2f}s/step')
    
    # Safety check
    if has_nan:
        print('⚠ NaN detected! Stopping training.')
        break

print(f'\n✓ Training complete! {global_step} steps')
wandb.finish()

In [ ]:
# Cell 10: Save Checkpoint
checkpoint = {
    'step': global_step,
    'predictor': predictor.state_dict(),
    'proj_in': model.proj_in.state_dict(),
    'pred_proj': model.pred_proj.state_dict(),
    'text_model': text_model.state_dict(),
    'text_proj': model.text_proj.state_dict(),
    'temperature': model.temperature.data,
    'optimizer': optimizer.state_dict(),
}

torch.save(checkpoint, 'vljepa_2gpu_wandb.pt')
print('✓ Saved checkpoint to vljepa_2gpu_wandb.pt')

# Also save to wandb
wandb.save('vljepa_2gpu_wandb.pt')
print('✓ Uploaded checkpoint to wandb')